# Mutation → Mechanism → Therapy — Main Run Notebook

**Repo:** [github.com/lightflow16/mutation-mechanism-therapy-amd](https://github.com/lightflow16/mutation-mechanism-therapy-amd)

**Clone once** — run the Python clone cell below (section 0). Do **not** paste `git clone ...` as bare Python; that causes `SyntaxError`.

AMD notebook: start in `/workspace`. Colab: start in `/content`.

## 0 — Clone + confirm repo root

In [ ]:
from pathlib import Path
import os, subprocess

ROOT = Path(".").resolve()
repo_dir = ROOT / "mutation-mechanism-therapy-amd"

if (ROOT / "src" / "pipeline.py").exists():
    print("Already in repo root:", ROOT)
elif (repo_dir / "src" / "pipeline.py").exists():
    os.chdir(repo_dir)
    print("Entered clone:", Path(".").resolve())
else:
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/lightflow16/mutation-mechanism-therapy-amd.git",
        ],
        check=True,
        cwd=ROOT,
    )
    os.chdir(repo_dir)
    print("Cloned to:", Path(".").resolve())

In [ ]:
# Clone once in terminal (not here), then open this notebook from the repo root:
#   cd /workspace && git clone --depth 1 https://github.com/lightflow16/mutation-mechanism-therapy-amd.git
#   cd mutation-mechanism-therapy-amd

from pathlib import Path
ROOT = Path('.').resolve()
assert (ROOT / 'src' / 'pipeline.py').exists(), (
    'Run from repo root. Clone: git clone https://github.com/lightflow16/mutation-mechanism-therapy-amd.git'
)
print('ROOT =', ROOT)

## 1 — Environment + paths (CPU ok, no GPU yet)

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path('.').resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.config import configure_paths, is_rocm, setup_env
from src.serving import verify_gpu_torch

configure_paths()
setup_env()

import torch
print('torch', torch.__version__)
print('platform:', 'ROCm/AMD' if is_rocm() else 'CUDA (Colab) or local')
print('LLM_BACKEND', os.environ.get('LLM_BACKEND', 'auto'))
print('GPU verify (CPU ok if not attached yet):', verify_gpu_torch())

In [ ]:
!pip install -q -r requirements.txt openai

In [ ]:
!bash scripts/setup_external.sh

## 2 — Instant demo (cached traces, NO GPU, NO vLLM)

In [ ]:
from src.pipeline import run_case
import json

for gene, mut in [('EGFR','L858R'), ('PIK3CA','E545K'), ('TP53','R175H')]:
    r = run_case(gene, mut, architecture='blackboard', use_cached_trace=True, live_evidence=False)
    tr = r['reasoning'].get('target_reasoning', r['reasoning'])
    print(gene, mut, '->', r['route'], '| therapies:', tr.get('therapy',{}).get('sensitivity'))

## 2b — MTB panel (cached blackboard trace, CPU only)

Prints the Molecular Tumor Board view: each agent's stance from `trace_*_blackboard.json`.

In [ ]:
from src.config import metrics_dir
from src.mtb_panel import format_mtb_panel
from pathlib import Path

# Prefer live metrics dir; fall back to repo cached trace
candidates = [
    metrics_dir() / 'trace_PIK3CA_E545K_blackboard.json',
    Path('data/traces/PIK3CA_E545K_blackboard.json'),
]
for p in candidates:
    if p.is_file():
        print(format_mtb_panel(p))
        break
else:
    print('No PIK3CA blackboard trace found — run section 2 or 3 first.')

## 2c — Reasoning ablation table (CPU, from saved metrics)

In [ ]:
import pandas as pd
from pathlib import Path
from src.config import metrics_dir

md = metrics_dir()
abl = md / 'ablation_results.csv'
ror = md / 'return_on_reasoning.csv'
if abl.is_file():
    display(pd.read_csv(abl))
else:
    print('No ablation_results.csv — run eval or full submission first.')
if ror.is_file():
    print('\nReturn on Reasoning:')
    display(pd.read_csv(ror))
llm = md / 'llm_calls.jsonl'
if llm.is_file():
    rows = [__import__('json').loads(l) for l in llm.read_text().strip().split('\n') if l]
    df = pd.DataFrame(rows)
    if 'multimodal_image' in df.columns:
        print('\nMultimodal calls:', df.groupby(['architecture','multimodal_image']).size())
    print('\nToken rollup by architecture:')
    display(df.groupby('architecture')[['ingress_tokens','egress_tokens','reasoning_tokens']].sum())

## 2d — Unknown / VUS demo (EGFR G719S — abstention path)

In [ ]:
from src.config import load_config, get_target
from src.pipeline import run_case
from src.variant_router import route_variant
from src.evidence import gather_evidence
from src.mtb_panel import format_vus_summary

cfg = load_config()
gene, mut = cfg['pipeline']['vus_demo_cases'][0]
target = get_target(cfg, gene, mut)
# CPU cached replay; set live_evidence=True for one GPU VUS run
r = run_case(gene, mut, architecture=cfg['pipeline'].get('vus_demo_architecture', 'single'),
             use_cached_trace=True, live_evidence=False)
routing = route_variant(target, r.get('structure', {}), r.get('evidence', []))
print('Routing:', routing)
tr = r['reasoning'] if isinstance(r.get('reasoning'), dict) else {}
print(format_vus_summary(tr, routing))

## 3 — Full GPU run (all flows, always)

One command runs **everything** on Colab or AMD:
- cached baseline + **live** single / cot / blackboard on all 3 demo mutations
- per-mutation architecture comparison + eval
- full TP53 rescue (ThermoMPNN + MPNN + ESMFold + Boltz)
- exports **all metrics** (calls.csv, phases.csv, llm_calls.jsonl, system_samples.csv, traces, comparisons)

**AMD:** transformers only (never `pip install vllm`). **Colab:** attach GPU before this section.

In [ ]:
import time
GPU_ATTACH_T0 = time.time()  # budget proxy (~240 min)
print('GPU attached at', GPU_ATTACH_T0)

In [ ]:
from src.config import is_rocm
from src.serving import platform_summary, verify_gpu_torch
from src.llm_client import use_vllm

print(platform_summary())
gpu = verify_gpu_torch()
assert gpu["ok"], f"GPU broken: {gpu.get('error')} — restart session if you ran pip install vllm on AMD"
print("LLM routing:", "vllm" if use_vllm() else "transformers")

In [ ]:
!bash scripts/setup_boltz_venv.sh
import os
from pathlib import Path
boltz = Path('external/boltz_venv/bin/boltz')
assert boltz.is_file(), 'Boltz missing — run setup_external.sh'
os.environ['BOLTZ_BIN'] = str(boltz.resolve())
print('BOLTZ_BIN', os.environ['BOLTZ_BIN'])

## 3b — Teacher–student LoRA (distill blackboard traces → train → infer)

In [ ]:
# Run AFTER blackboard traces exist (section 3 or data/traces/)
!PYTHONPATH=. python train/build_dataset.py --from-traces
!PYTHONPATH=. python train/lora_sft.py
print('LoRA adapter ready at /workspace/shared/lora_adapter_final')

In [ ]:
# LoRA training (~10-30 min GPU) — run before full submission if using adapter
# Ignore torchao 'Failed to load cutlass/mxfp8' on Colab (bf16 LoRA does not use them)
!pip install -q 'torchao>=0.16.0'
!PYTHONPATH=. python train/lora_sft.py

In [ ]:
# FULL SUBMISSION — all architectures × all cases + eval + metrics bundle
import os
import sys
from pathlib import Path

# Fresh src after git pull (fixes stale metrics.log_llm_call signature on Colab)
for _m in list(sys.modules):
    if _m == 'src' or _m.startswith('src.'):
        del sys.modules[_m]

from src.submission import run_full_submission

LORA = os.environ.get('LORA_PATH', '/workspace/shared/lora_adapter_final')
if not os.path.isdir(LORA):
    alt = Path('.').resolve() / 'shared' / 'lora_adapter_final'
    LORA = str(alt) if alt.is_dir() else None

def _lora_ok(path):
    if not path: return False
    p = Path(path)
    return p.is_dir() and any(p.glob('adapter_model.*')) or any(p.glob('*.safetensors'))

print('LoRA path:', LORA, '| weights found:', _lora_ok(LORA))
print('Live console echo enabled (pipeline.verbose_progress in targets.yaml)')

report = run_full_submission(lora_path=LORA)
print('Metrics bundle:', report.get('metrics_bundle'))

In [ ]:
# (Optional) CLI equivalent:
# !PYTHONPATH=. python scripts/run_full_submission.py --lora-path /workspace/shared/lora_adapter_final

In [ ]:
# TP53 rescue is included in full submission (run_mutation_comparison).
# Uncomment only for an isolated rescue debug run:
# !pip install -q pytorch-lightning torchmetrics omegaconf wandb tqdm
# from src.pipeline import run_mutation_comparison
# run_mutation_comparison('TP53', 'R175H', use_cached_trace=False, run_rescue_branch=True)

## 5 — Multimodal ablation (from llm_calls.jsonl)

In [ ]:
import pandas as pd, json
from src.config import metrics_dir
llm = metrics_dir() / 'llm_calls.jsonl'
if llm.is_file():
    rows = [json.loads(l) for l in llm.read_text().splitlines() if l.strip()]
    df = pd.DataFrame(rows)
    if 'multimodal_image' in df.columns:
        print('Multimodal ablation:')
        display(df.groupby(['architecture','multimodal_image'])[['total_tokens','latency_s']].mean())
else:
    print('Run full submission first')

## 6 — Agent autonomy report (CPU)

In [ ]:
import json
import pandas as pd
from src.config import metrics_dir

md = metrics_dir()
p = md / 'autonomy_report.json'
if not p.is_file():
    print('Run full submission first (trust_eval step)')
else:
    rep = json.loads(p.read_text())
    print('§10 Agent Autonomy (MMT-R)')
    print('  workflow_completion_rate:', rep.get('workflow_completion_rate'))
    print('  task_pass_rate:', rep.get('task_pass_rate'))
    print('  DBTL claim:', rep.get('dbtl_claim'))
    dbtl = rep.get('dbtl_level3_tp53') or {}
    print('  DBTL L3 success:', dbtl.get('dbtl_success'), '| designs:', dbtl.get('n_designs'))
    print('  Refusal rate:', rep.get('refusal_rate'))
    print('  TEVV constraint pass rate:', rep.get('tevv_constraint_pass_rate'))
    print('\nTask suite:')
    display(pd.DataFrame(rep.get('task_suite', [])))
    print('\nABLE metrics:')
    display(pd.DataFrame(rep.get('able_metrics', [])))
    if (md / 'tevv_lite.csv').is_file():
        print('\nTEVV-lite (constraints + refusal):')
        display(pd.read_csv(md / 'tevv_lite.csv'))
    print('\nTrait grid (sample):')
    display(pd.DataFrame(rep.get('trait_grid', [])).head(8))

In [ ]:
# Open workflow dashboard (productive throughput + agent trace timeline)
from IPython.display import IFrame
from src.config import metrics_dir
dash = metrics_dir() / 'workflow_trace_dashboard.html'
if dash.is_file():
    display(IFrame(src=str(dash), width='100%', height=520))
else:
    print('Run full submission first, or: PYTHONPATH=. python scripts/generate_workflow_report.py')

## 4 — Download metrics + Colab vs AMD comparison

## 8 — Fold confidence figures

In [ ]:
!PYTHONPATH=. python scripts/plot_confidence_benchmark.py
from IPython.display import SVG
from src.config import metrics_dir
svg = metrics_dir() / 'confidence_scatter_F1.svg'
if svg.is_file():
    display(SVG(filename=str(svg)))
else:
    print('No benchmark_confidence.csv yet')

In [ ]:
from pathlib import Path
from src.config import metrics_dir, shared_dir, configure_paths
from src.metrics_bundle import export_metrics_bundle

configure_paths()
md = metrics_dir()
shared = shared_dir()
bundle = export_metrics_bundle(shared / 'metrics_bundle_latest.tgz')
print('Bundle:', bundle)
print('Metrics dir:', md)
for name in sorted(md.glob('*')):
    if name.is_file():
        print(' ', name.name, name.stat().st_size)

# Colab: download bundle to laptop
try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    print('On AMD: copy from', bundle)